## Setup


In [2]:
import pandas as pd
import numpy as np
# from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
# from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [4]:

df_model = pd.read_csv('../data/clean/model_ready_data.csv')
print(df_model.head())

   Delivery_person_Age  Delivery_person_Ratings Weatherconditions  \
0                   37                      4.9             Sunny   
1                   34                      4.5            Stormy   
2                   23                      4.4        Sandstorms   
3                   38                      4.7             Sunny   
4                   32                      4.6            Cloudy   

  Road_traffic_density  Vehicle_condition Type_of_order Type_of_vehicle  \
0                 High                  2         Snack      motorcycle   
1                  Jam                  2         Snack         scooter   
2                  Low                  0        Drinks      motorcycle   
3               Medium                  0        Buffet      motorcycle   
4                 High                  1         Snack         scooter   

   multiple_deliveries Festival           City  Delivery_Time_min  \
0                    0       No          Urban                 24

**upload file model_ready_data using google.colab files**

In [3]:
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df_model = pd.read_csv(file_name)

print("Dataset shape:", df_model.shape)
print(df_model.head())
print(df_model.columns.tolist())


Saving model_ready_data.csv to model_ready_data.csv
Dataset shape: (41872, 14)
   Delivery_person_Age  Delivery_person_Ratings Weatherconditions  \
0                   37                      4.9             Sunny   
1                   34                      4.5            Stormy   
2                   23                      4.4        Sandstorms   
3                   38                      4.7             Sunny   
4                   32                      4.6            Cloudy   

  Road_traffic_density  Vehicle_condition Type_of_order Type_of_vehicle  \
0                 High                  2         Snack      motorcycle   
1                  Jam                  2         Snack         scooter   
2                  Low                  0        Drinks      motorcycle   
3               Medium                  0        Buffet      motorcycle   
4                 High                  1         Snack         scooter   

   multiple_deliveries Festival           City  Deliver

## Define the target and features

In [5]:
df_model.columns = df_model.columns.str.strip()
target = "Delivery_Time_min"

numeric_features = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Vehicle_condition",
    "multiple_deliveries",
    "Distance_km",
    "Order_Hour",
    "Is_Weekend"
]

categorical_features = [
    "Weatherconditions",
    "Road_traffic_density",
    "Type_of_order",
    "Type_of_vehicle",
    "Festival",
    "City"
]

required_columns = (
    numeric_features + categorical_features + [target]
)

missing_columns = [ col for col in required_columns if col not in df_model.columns]

if missing_columns:
    raise ValueError(
        f"Missing columns in the dataset: {missing_columns}"
    )

print('All Columns are availables')



All Columns are availables


## Check and convert data types
- The target must be numeric for regression
- Categorical columns must be treated as categories
- Rows with a missing target cannot be used because the real delivery time is unknown

In [6]:
# convert numeric
for col in numeric_features + [target]:
  df_model[col] = pd.to_numeric(df_model[col],errors="coerce")

# convert category
for col in categorical_features:
  df_model[col] = df_model[col].astype('string')

# target can't be imputed
df_model = df_model.dropna(subset=[target]).reset_index()

# check missing values again
print(f'messing values: {df_model.isna().sum()}')
# check data types
print(f'datatypes: {df_model.dtypes}')


messing values: index                      0
Delivery_person_Age        0
Delivery_person_Ratings    0
Weatherconditions          0
Road_traffic_density       0
Vehicle_condition          0
Type_of_order              0
Type_of_vehicle            0
multiple_deliveries        0
Festival                   0
City                       0
Delivery_Time_min          0
Distance_km                0
Order_Hour                 0
Is_Weekend                 0
dtype: int64
datatypes: index                        int64
Delivery_person_Age          int64
Delivery_person_Ratings    float64
Weatherconditions           string
Road_traffic_density        string
Vehicle_condition            int64
Type_of_order               string
Type_of_vehicle             string
multiple_deliveries          int64
Festival                    string
City                        string
Delivery_Time_min            int64
Distance_km                float64
Order_Hour                   int64
Is_Weekend                   int64


## Separate X and y then split the data:
- 80% of the data is used to learn the model
- 20% is kept for unbiased testing
- random_state = 42 make the split reproducible
  
The dataset is divided into 80% training data and 20% testing data. The training set is used to learn the relationship between the input features and the target variable, while the test set is kept separate to evaluate how well the models perform on unseen data. random_state=42 is used to ensure that the same split can be reproduced.

In [7]:

x = df_model[numeric_features + categorical_features]
y = df_model[target]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
    )

print("Training features: ",x_train.shape)
print("Training target: ",y_train.shape)
print("Testing features: ",x_test.shape)
print("Testing target: ",y_test.shape)

Training features:  (33497, 13)
Training target:  (33497,)
Testing features:  (8375, 13)
Testing target:  (8375,)


## Create the preprocessing pipeline

- Categorical variables cannot be directly used by most machine-learning regression models. Therefore, OneHotEncoder is used to convert categorical variables into numerical binary features. The option handle_unknown='ignore' ensures that the model can handle categories that may appear in the test data but were not present during training.
- StandardScaler is applied to the numerical variables to put them on a comparable scale. This is particularly important for models such as Linear Regression and SVR, which can be sensitive to differences in feature scales. Standardization transforms the numerical variables based on their mean and standard deviation.

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numeric_features)
    ]
)

## Define 4 Models

Four different regression models are evaluated: Linear Regression, Support Vector Regression (SVR), Random Forest Regression, and Gradient Boosting Regression. These models represent different modeling approaches, including linear and non-linear methods. Comparing several models allows us to determine which approach performs best for this dataset rather than assuming that one specific algorithm will be optimal.

In [9]:
models = {
    "Linear Regression":LinearRegression(),
    "SVR":SVR(kernel = 'rbf'),

    "Random Forest":RandomForestRegressor(n_estimators=100,random_state=42, n_jobs=-1),
    "Gradient Boosting":GradientBoostingRegressor(n_estimators=100, random_state=42)
}




## Train & Evaluate All Models

- **Random Forest** has the lowest MAE (3.18) and RMSE (4.03), meaning its predictions have the smallest errors.
- It also has the highest R² (0.81), meaning it explains about 81% of the variation in the target variable in this test set.
- **Gradient Boosting** is the second-strongest model with an R² of 0.76.
- **SVR** performs moderately well with an R² of 0.73.
- **Linear Regression** performs weakest, with an R² of 0.58 and the highest prediction errors.

- **conclusion:** Based on these test-set metrics, **Random Forest** is the most suitable model of the four for this dataset, while Linear Regression provides the weakest predictive performance.
- **Random Forest** was selected as the final model because it achieved the lowest **MAE** and **RMSE** and the highest **R²** among the evaluated models on the test dataset.

In [11]:
results = []
for name,model in models.items():
  # create full pipeline
  pipe = Pipeline(steps=[
      ('preprocessor',preprocessor),
      ('model',model)

  ])
  # train
  pipe.fit(x_train,y_train)

  # Predict
  y_pred = pipe.predict(x_test)

  # Metrices
  mae = mean_absolute_error(y_test,y_pred)
  mse = mean_squared_error(y_test,y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test,y_pred)

  results.append({
      "Model":name,
      "MAE":round(mae,2),
      "MSE":round(mse,2),
      "RMSE":round(rmse,2),
      "R2":round(r2,2)

  })

results_df = pd.DataFrame(results)
print(results_df.sort_values(by="RMSE"))



               Model   MAE    MSE  RMSE    R2
2      Random Forest  3.18  16.23  4.03  0.81
3  Gradient Boosting  3.59  20.59  4.54  0.76
1                SVR  3.78  23.42  4.84  0.73
0  Linear Regression  4.82  37.00  6.08  0.58


- Linear Regression: Fast but usually lower performance on complex data.
- Random Forest: Good balance, reduces overfitting, usually strong performer.
- Gradient Boosting: Often gives the best accuracy but slower to train.

We will select the model with the lowest RMSE and highest R² as our best model. (Random Forest)

## 5-Fold Cross-Validation

In [14]:
from sklearn.model_selection import cross_val_score

cv_results = []

for name, model in models.items():

    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    scores = cross_val_score(
        pipe,
        x_train,
        y_train,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )

    # Convert negative MSE back to positive MSE
    mse_scores = -scores

    # Convert MSE to RMSE
    rmse_scores = np.sqrt(mse_scores)

    # cross validation R2
    r2_scores = cross_val_score(
        pipe,
        x_train,
        y_train,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "CV RMSE Mean": round(rmse_scores.mean(), 2),
        "CV RMSE Std": round(rmse_scores.std(), 2),
        "CV R2 Mean": round(r2_scores.mean(), 2),
        "CV R2 Std": round(r2_scores.std(), 2)
    })

    

cv_results_df = pd.DataFrame(cv_results)

print(cv_results_df.sort_values(by="CV RMSE Mean"))

               Model  CV RMSE Mean  CV RMSE Std  CV R2 Mean  CV R2 Std
2      Random Forest          4.07         0.02        0.81       0.00
3  Gradient Boosting          4.54         0.04        0.77       0.01
1                SVR          4.88         0.03        0.73       0.00
0  Linear Regression          6.02         0.04        0.59       0.00
